# Error Catching 與 Exception Handling 從零開始

本筆記從零開始介紹 Python 的錯誤捕捉（error catching）與例外處理（exception handling），包含概念、哲學、語法與實作案例。最後附上練習題。

參考資料：
- Python 官方教學（繁中）：https://docs.python.org/zh-tw/3/tutorial/errors.html
- Python 官方教學（英文）：https://docs.python.org/3/tutorial/errors.html
- *Clean Code* 相關章節（PDF）：https://ptgmedia.pearsoncmg.com/images/9780132350884/samplepages/9780132350884.pdf
- 《代碼整潔之道》中文 PDF：https://awesome-programming-books.github.io/clean-code/%E4%BB%A3%E7%A0%81%E6%95%B4%E6%B4%81%E4%B9%8B%E9%81%93.pdf


## 0. 練習題

1. （APCS）讀入兩個整數 a, b，若有任一非整數則提示錯誤並結束；否則輸出它們的最大公因數。
2. 讀入 N 筆成績，若有分數不在 0~100，提示「資料錯誤」並忽略該筆。最後輸出有效成績平均。
3. 讀入一行測資（空白分隔整數），若含非整數，顯示錯誤並結束。
4. 讀入矩陣大小 n，若 n 不是正整數則要求重新輸入，直到合法為止。
5. 讀入多行測資直到 EOF，每行必須是兩個整數 a b。若格式錯誤，顯示該行錯誤並略過。
6. 自訂 `InvalidIDError`，學號必須是 8 位數字；不合法就丟出例外並提示使用者。
7. 寫一個函式 `safe_divide(a, b)`：b = 0 時丟出 `ZeroDivisionError`，並在呼叫端正確處理。
8. 寫一個排序程式：若輸入序列長度與宣告的 N 不一致，丟出自訂例外並終止。


## 1. 最簡單的錯誤示例

下面這行會直接出錯，因為 `b` 沒有定義。


In [ ]:
a = 10
print(a + b)  # NameError: name 'b' is not defined


## 2. 為什麼需要例外處理？

程式在執行時可能遇到各種「預期內」與「預期外」的狀況：

- 使用者輸入不是數字
- 檔案不存在
- 除以零
- 網路斷線

若沒有例外處理，程式一旦出錯就會中止。例外處理讓我們能夠：

- 提供更清楚的錯誤訊息
- 讓程式保持穩定（不一定中止）
- 把「真正重要的錯誤」與「可恢復的錯誤」分開處理


## 3. 概念：Error vs Exception

- **Error（錯誤）**：廣義上的失敗情況
- **Exception（例外）**：Python 在執行時偵測到異常狀況會「丟出」的物件

重點：**例外是 Python 提供的一種機制**，不是只能用來「補救」，也能用來「控制流程」。


## 4. Error 與 Exception 有區別嗎？（重點整理）

### 1) 一句話版本
- **Error**：廣義的「錯誤」或失敗情況。
- **Exception**：Python 會拋出（raise）並可被捕捉（try/except）的「例外物件」。

### 2) 在 Python 裡的實際意義
- **Exception 是可以被程式處理的錯誤**（例如 `ValueError`、`ZeroDivisionError`）。
- **Error 通常代表程式碼有問題**（例如 `SyntaxError`、`IndentationError`），
  這類錯誤多半不是用 try/except 解決，而是直接修正程式碼。

### 3) 舉例
- `ValueError`：輸入不是整數 → 可以用 try/except 提示使用者重新輸入。
- `FileNotFoundError`：檔案不存在 → 可以提示路徑錯誤或改用預設檔。
- `SyntaxError`：少了冒號或括號 → 程式根本無法執行，必須修程式。

### 4) 實務判斷
- **能處理就捕捉**：例如輸入、檔案、網路等可恢復的狀況。
- **不能修復就拋出**：例如邏輯錯誤或不該發生的狀態，讓它及早爆掉。

### 5) 簡短結論
- **Exception 是 Error 的一種**（Python 可處理的那種）。
- **Error（語法/縮排等）多半只能修 code**，不是 try/except 可以補救。


## 5. try / except 語法詳解

### 基本結構
```python
try:
    # 可能出錯的程式碼
except SomeError:
    # 發生 SomeError 時要做的事
```

### 進階結構（含 else / finally）
```python
try:
    # 可能出錯
except ValueError:
    # 特定例外處理
except (TypeError, ZeroDivisionError):
    # 多個例外可用 tuple
else:
    # 沒有例外才執行
finally:
    # 一定會執行
```

### 語法重點
- `try` 必須搭配至少一個 `except` 或 `finally`。
- `except` 可以有多個，順序要「由精準到廣泛」。
- `except Exception as e` 可取得例外物件。
- `else` 只在沒有例外時執行。
- `finally` 無論是否例外都會執行，常用於釋放資源。


## 6. 最基本語法：try / except

### 範例：輸入數字並除以 2


In [ ]:
try:
    s = input("請輸入整數：")
    n = int(s)
    print(n / 2)
except ValueError:
    print("你輸入的不是整數")


說明：
- `try` 內的程式可能出錯
- `except ValueError` 只捕捉特定的例外


## 7. 多個 except

當可能出現不同錯誤時，可以分別處理。


In [ ]:
try:
    a = int(input("a = "))
    b = int(input("b = "))
    print(a / b)
except ValueError:
    print("請輸入整數")
except ZeroDivisionError:
    print("b 不能是 0")


## 8. 捕捉所有例外（不建議濫用）

`except Exception` 會抓到幾乎所有的例外。它可以用於「最後防線」，
但**不應該**用來掩蓋錯誤原因。


In [ ]:
try:
    lst = [1, 2, 3]
    print(lst[10])
except Exception as e:
    print("發生錯誤：", e)


## 9. else 與 finally

- `else`：只有在 **try 沒有發生錯誤** 時才執行
- `finally`：無論是否錯誤都會執行（常用於釋放資源）


In [ ]:
try:
    f = open("demo.txt", "w", encoding="utf-8")
    f.write("Hello")
except OSError:
    print("檔案操作失敗")
else:
    print("寫入成功")
finally:
    f.close()
    print("檔案已關閉")


## 10. raise：主動丟出例外

當你偵測到「不合理的狀態」時，應該主動丟出例外，
這可以讓錯誤更早被發現。


### 補充：`raise` 的用途與時機

`raise` 用來**主動拋出例外**，常見用途有：

- **輸入/參數驗證**：發現不合理的值就立刻中止，避免錯誤擴散。
- **自訂錯誤語意**：用自訂例外讓錯誤更可讀、更好查。
- **保留錯誤資訊**：在 except 中處理完後，必要時可以 `raise` 讓錯誤繼續往上拋。

小提醒：
- `raise` 後面可以接「例外類別」或「例外物件」。
- 在 `except` 區塊中直接 `raise`，會**重新拋出原本的例外**。


### 補充：`raise` 會發生什麼事？

- `raise` 會**立刻中斷目前函式的流程**，後面的 `return` 不會執行。
- 如果外層有對應的 `try/except`，就會跳到 `except` 繼續執行。
- 如果沒有被捕捉，例外會一路往上拋，程式可能終止。


In [ ]:
def f():
    raise ValueError('錯誤')
    return 123  # 永遠不會執行

try:
    f()
except ValueError:
    print('接到了')


In [ ]:
def set_age(age):
    if age < 0:
        raise ValueError("年齡不能是負數")
    return age

# print(set_age(10))
print(set_age(-5))


## 11. 自訂例外類別

當你的程式有特定領域規則時，自訂例外可以讓錯誤語意更清楚。


In [ ]:
class InvalidScoreError(Exception):
    pass

def set_score(score):
    if not (0 <= score <= 100):
        raise InvalidScoreError("分數必須在 0~100")
    return score

print(set_score(88))
# print(set_score(120))


## 12. 常見 Error / Exception 類型速覽

以下是初學者最常遇到的幾種錯誤：

- `SyntaxError`：語法錯誤，例如漏掉冒號或括號不成對。
- `IndentationError`：縮排錯誤，Python 對縮排非常敏感。
- `NameError`：使用了未定義的變數名稱。
- `TypeError`：型別不正確，例如把字串和整數相加。
- `ValueError`：型別對但值不合理，例如 `int('abc')`。
- `ZeroDivisionError`：除以 0。
- `IndexError`：索引超出範圍，例如 `lst[10]`。
- `KeyError`：字典不存在的 key，例如 `d['x']`。
- `FileNotFoundError`：檔案不存在。
- `EOFError`：在讀取輸入時遇到 EOF（例如檔案結束或輸入被中斷）。
- `OSError`：系統或檔案操作相關錯誤（更一般）。

提示：`SyntaxError` 與 `IndentationError` 是在「程式解析階段」就會出現，
通常不是用 `try/except` 解決，而是直接修正程式碼。

補充：`EOFError` 常見於讀取輸入或檔案時遇到結尾（EOF）。
在 Notebook 中無法用 Ctrl-D 送出 EOF，所以通常改用「空行結束」或「檔案讀取」示範。


### 範例：Notebook 版（用空行作為輸入結束）


In [ ]:
# Notebook 版本：用空行表示輸入結束
lines = []
while True:
    line = input()
    if line == '':
        break
    lines.append(line)

print('共讀到', len(lines), '行')


### 範例：檔案讀取版（遇到檔案結尾）


In [ ]:
# 先在同一資料夾準備 demo.txt
try:
    with open('demo.txt', 'r', encoding='utf-8') as f:
        while True:
            line = f.readline()
            if line == '':
                raise EOFError
            print(line.strip())
except FileNotFoundError:
    print('找不到 demo.txt')
except EOFError:
    print('讀到檔案結尾')


### 各種 Error / Exception 的最小範例

下面每一種都有最小示例；其中 `SyntaxError` 與 `IndentationError` 只示範程式碼樣子，
在 Notebook 直接執行會讓整個 cell 失敗，請只看範例。

- `SyntaxError`：
  ```python
  if True
      print('hi')
  ```
- `IndentationError`：
  ```python
  def f():
  print('hi')
  ```


**NameError 範例**


In [ ]:
a = 1
print(a + b)  # b 未定義


**TypeError 範例**


In [ ]:
print('1' + 2)  # 字串 + 整數


**ValueError 範例**


In [ ]:
int('abc')  # 無法轉成整數


**ZeroDivisionError 範例**


In [ ]:
print(1 / 0)


**IndexError 範例**


In [ ]:
lst = [1, 2, 3]
print(lst[10])


**KeyError 範例**


In [ ]:
d = {'a': 1}
print(d['b'])


**FileNotFoundError 範例**


In [ ]:
open('no_such_file.txt', 'r', encoding='utf-8')


**OSError 範例**


In [ ]:
open('', 'r')


**EOFError 範例**


In [ ]:
# 手動模擬 EOFError
raise EOFError


**補充：為什麼 `OSError` 會印出 `FileNotFoundError`？**

- `FileNotFoundError` 是 `OSError` 的子類別。
- 所以 `open('', 'r')` 其實拋出的是更具體的 `FileNotFoundError`。
- 若要示範「一般的 OSError」，可以說明：`OSError` 是系統/檔案相關錯誤的總類，
  實際發生時常會是其子類別（例如 `FileNotFoundError`、`PermissionError`、`IsADirectoryError`）。


## 13. 例外處理的哲學（簡化版）

從 Clean Code 與官方文件的精神整理：

- **不要忽略例外**：不要用空的 `except: pass`
- **例外訊息要清楚**：讓未來的你或同事能快速理解
- **只處理你能處理的錯誤**：不能修復就讓它往上拋
- **讓錯誤更早被發現**：與其默默修補，不如清楚失敗

這些原則能讓程式更可靠、也更容易維護。


## 14. 經典案例：輸入驗證

需求：讀入兩個整數，輸出 `a / b`，要求：
- 輸入不是整數要提示
- b = 0 要提示
- 成功才顯示結果


In [ ]:
try:
    a = int(input("a = "))
    b = int(input("b = "))
    result = a / b
except ValueError:
    print("請輸入整數")
except ZeroDivisionError:
    print("b 不能為 0")
else:
    print("結果：", result)


## 15. 常見陷阱

- **except 太大包**：會把真正的錯誤藏起來
- **錯誤訊息不清楚**：不利於維護
- **try 區塊太大**：應只包住可能出錯的那幾行


## 15.5 try 區塊要小、except 要精準（對比）

重點：只包住「可能出錯」的幾行，並且針對可處理的例外做處理。


In [ ]:
# 不建議：try 太大、except 太泛
try:
    name = input('name = ')
    age = int(input('age = '))
    f = open('log.txt', 'a', encoding='utf-8')
    f.write(name + ':' + str(age) + '\n')
    f.close()
    print('OK')
except Exception:
    print('錯誤')

# 建議：縮小 try 範圍、精準處理
name = input('name = ')
try:
    age = int(input('age = '))
except ValueError:
    print('年齡必須是整數')
    raise

try:
    with open('log.txt', 'a', encoding='utf-8') as f:
        f.write(name + ':' + str(age) + '\n')
except OSError:
    print('檔案寫入失敗')


## 16. Fix-it 改錯練習

下面每題都有錯誤，請找出並修正，讓程式可以正常運作。


**題目 1 提示：** 提示：錯不在除以 0，而是輸入轉整數可能失敗。想想 `int()` 會丟什麼例外。


In [ ]:
# 1) 輸入整數除法：輸入不是整數時會爆掉
try:
    n = int(input('n = '))
    print(10 / n)
except ZeroDivisionError:
    print('n 不能為 0')
    


**題目 2 提示：** 提示：`open()` 可能失敗，先處理檔案不存在的情況。


In [ ]:

# 2) 檔案讀取：若檔案不存在會爆掉
f = open('data.txt', 'r', encoding='utf-8')
print(f.read())
f.close()



**題目 3 提示：** 提示：`input()` 是字串，不能直接做除法。


In [ ]:


# 3) 型別錯誤：輸入不是整數時會爆掉
try:
    a = input('a = ')
    b = input('b = ')
    print(a / b)
except ValueError:
    print('請輸入整數')

    


**題目 4 提示：** 提示：try 區塊太大，請拆小並分別處理可預期的錯誤。


In [ ]:

# 4) try 區塊太大：任何錯誤都被同一個訊息吃掉
try:
    name = input('name = ')
    age = int(input('age = '))
    with open('log.txt', 'a', encoding='utf-8') as f:
        f.write(name + ':' + str(age) + '\n')
    print('OK')
except Exception:
    print('錯誤')
    


**題目 5 提示：** 提示：自訂例外必須繼承 `Exception`。


In [ ]:

# 5) 自訂例外：使用錯誤的繼承關係
class InvalidScoreError:
    pass

def set_score(score):
    if not (0 <= score <= 100):
        raise InvalidScoreError('分數必須在 0~100')
    return score
